# Laboratorio 3: Sistema de Búsqueda
## Estudio experimental del escalamiento de listas, árboles binarios de búsqueda y árboles B+

- **Curso:** Estructuras de datos y laboratorio.
- **Autor:** Daniel Sánchez Escobar 
- **Fecha:** [fecha de entrega]  

### Resumen
Este informe estudia experimentalmente cómo cambia el tiempo de ejecución de un sistema de
administración de estudiantes según la estructura de datos que lo soporta: una lista de Python,
un árbol binario de búsqueda (ABB) y un árbol B+. Se miden las operaciones de búsqueda por ID,
inserción, listado ordenado y búsqueda por rango, variando el tamaño de los datos (N), el orden
de inserción (aleatorio u ordenado) y otros parámetros. Los resultados se comparan con la
complejidad teórica de cada estructura.

**Hallazgos principales:** [se completa al final, con los resultados obtenidos].

> **Uso de IA:** este trabajo se desarrolló con asistencia de herramientas de IA, autorizada por el
> enunciado del laboratorio. El detalle está en la sección final "Uso de IA y código de honor".

## 1. Problema y objetivo

### El problema
Una institución educativa necesita administrar los registros de sus estudiantes. Cada registro
contiene un **ID** único de matrícula, un **nombre**, una **edad** y un **promedio** académico.
El sistema debe permitir:

1. **Buscar** un estudiante a partir de su ID.
2. **Insertar** un nuevo estudiante.
3. **Listar** todos los estudiantes en orden ascendente de ID.

Además de las tres operaciones del enunciado, se implementó la **búsqueda por rango de ID**
(todos los estudiantes con ID entre `lb` y `ub`), que permite comparar las estructuras en una
operación donde el orden de los datos es especialmente relevante.

Los registros usados en los experimentos son **sintéticos**: los IDs, edades y promedios se
generan con semillas fijas (reproducibles) y los nombres se toman de un conjunto generado con
la librería Faker.

### Objetivo
Estudiar experimentalmente cómo las distintas estrategias de almacenamiento y búsqueda
(lista, ABB y árbol B+) afectan el tiempo de ejecución a medida que crece el tamaño de los
datos, y contrastar los resultados con la complejidad teórica. Las conclusiones se sustentan
en mediciones repetidas y análisis estadístico.

### Preguntas que guían el estudio
- ¿Los tiempos medidos se comportan como predice la complejidad teórica?
- ¿Qué ocurre con el ABB cuando los datos se insertan ordenadamente, y qué relación hay entre la altura del árbol y el tiempo de búsqueda?
- ¿Qué diferencias aparecen entre datos aleatorios y ordenados?
- ¿A partir de qué tamaño de entrada se vuelven claramente visibles las diferencias entre estructuras?
- ¿Existen costos constantes que hagan que algoritmos de distinta complejidad tengan tiempos similares con entradas pequeñas?

### Variables del estudio
| Variable | Valores |
|---|---|
| Estructura | Lista, ABB, árbol B+ |
| Tamaño de los datos (N) | [valores finales] |
| Número de búsquedas por medición (M) | [valor] |
| Orden de inserción | IDs aleatorios, IDs ordenados |
| Tamaño del resultado en rangos (k) | [valor(es)] |
| Orden del árbol B+ | [valor(es)] |

## 2. Estructuras de datos y algoritmos

Las tres estructuras implementan la misma interfaz (clase abstracta `Estructura`) con las
operaciones `insertar`, `buscar`, `listar` y `buscar_rango_id`, de modo que las mediciones
comparan implementaciones intercambiables. Las clases están en archivos `.py` del repositorio
y se importan en este notebook.

**Notación:** 
- N = número de estudiantes almacenados 
- h = altura del árbol
- k = número de estudiantes que devuelve una consulta por rango
- n = orden del árbol B+.

### 2.1 Lista de Python (`GestorListaEstudiantes`)
Los estudiantes se guardan en una lista nativa, en el orden en que llegan.
- **Insertar:** `append` al final, sin comparar con los elementos existentes.
- **Buscar:** recorrido lineal hasta encontrar el ID (o terminar la lista si no existe).
- **Listar:** como la lista no está ordenada, se ordena una copia con `sorted` (Timsort) y luego
  se recorre. Timsort aprovecha los tramos ya ordenados, así que con datos ya ordenados cuesta
  cerca de O(N).
- **Rango:** se recorre toda la lista filtrando los IDs dentro de [lb, ub] y solo se ordenan las
  coincidencias.

### 2.2 Árbol binario de búsqueda (`ABB`)
Cada nodo guarda un estudiante y referencias a sus hijos izquierdo y derecho. Los IDs menores o
iguales van a la izquierda y los mayores a la derecha. El árbol **no se balancea**, su forma
depende del orden de inserción. Con IDs aleatorios la altura esperada es de orden O(log N), mientras que con IDs
insertados en orden creciente el árbol se degenera en una cadena de altura N.

Los algoritmos de **inserción** y **búsqueda** son las versiones iterativas presentadas en
*Introduction to Algorithms* (Cormen, Leiserson, Rivest y Stein), traducidas a Python. El
**recorrido in-orden** para listar parte del algoritmo del mismo libro, que es recursivo, pero aquí se
implementó de forma **iterativa con pila explícita**.

- **Insertar y buscar:** descenso iterativo desde la raíz, con costo proporcional a la altura h.
- **Listar:** recorrido in-orden iterativo con pila explícita (evita el límite de recursión de
  Python en árboles muy profundos), que visita los IDs en orden ascendente.
- **Rango:** descenso a la primera clave ≥ lb y recorrido in-orden, descartando subárboles
  fuera del rango y deteniéndose al superar ub.
- **Altura:** recorrido por niveles, usado en el experimento que relaciona altura y tiempo.

### 2.3 Árbol B+ (`ArbolBPlus`)
Árbol balanceado de orden n: cada nodo tiene como máximo n punteros y n−1 claves. Los datos
están solo en las **hojas**, que además se enlazan entre sí en una lista ligada. Los nodos
internos solo guardan claves de búsqueda. Todas las hojas están a la misma profundidad.

Los algoritmos de **búsqueda**, **inserción** (con división de hojas y nodos internos) y
**búsqueda por rango** son los presentados en *Database System Concepts* (Silberschatz, Korth y Sudarshan), traducidos a Python.

- **Insertar:** se desciende a la hoja correspondiente. Si está llena, se divide en dos y la
  clave separadora sube al padre (que puede dividirse a su vez, hasta llegar a crear una nueva raíz).
- **Buscar:** descenso desde la raíz hasta la hoja y búsqueda de la clave dentro de ella. En cada
  nodo la búsqueda es lineal, así que el costo también depende del orden n.
- **Listar:** se baja a la hoja más a la izquierda y se recorren las hojas enlazadas, sin volver
  a subir por el árbol.
- **Rango:** descenso hasta la hoja de `lb` y recorrido de las hojas enlazadas hasta superar `ub`.

### 2.4 Complejidad teórica (la predicción que se contrastará con los datos)
| Estructura | Insertar | Buscar | Listar | Rango |
|---|---|---|---|---|
| Lista | O(1) | O(N) para búsqueda lineal | O(N log N) debido al ordenamiento previo | O(N + k log k) |
| ABB, IDs aleatorios | O(log N) esperado (equilibrado) | O(log N) esperado (equilibrado) | O(N) (recorrer el total de nodos)| O(log N + k) esperado |
| ABB, IDs ordenados | O(N) por inserción (O(N²) construir todo) | O(N) | O(N) | O(N + k) |
| Árbol B+ (orden fijo) | O(log N) | O(log N) | O(N) | O(log N + k) |

A continuación, en cada experimento se comprueba si los tiempos
medidos la respaldan o no.

## 3. Entorno de ejecución

Todos los tiempos de este informe se midieron en el equipo y con el software descritos a
continuación. Los resultados son válidos para este entorno: en otro equipo cambiarán los
valores absolutos, aunque no necesariamente las tendencias. Las versiones exactas de las
librerías están en `requirements.txt`.

In [21]:
import sys
import platform
import subprocess
from importlib import metadata

import psutil


def nombre_cpu() -> str:
    """
    Devuelve el nombre comercial del procesador. platform.processor() suele venir
    vacío o genérico, así que se consulta según el sistema operativo: registro de
    Windows, sysctl en macOS o /proc/cpuinfo en Linux.

    Origen: generada con asistencia de IA (Claude, Anthropic, octubre 2026),
    autorizado por el enunciado del Laboratorio 3.
    Verificación: se validó que efectivamente coincide con las características del sistema.
    """
    try:
        if platform.system() == "Windows":
            import winreg
            clave = winreg.OpenKey(winreg.HKEY_LOCAL_MACHINE,
                                   r"HARDWARE\DESCRIPTION\System\CentralProcessor\0")
            return winreg.QueryValueEx(clave, "ProcessorNameString")[0].strip()
        if platform.system() == "Darwin":
            return subprocess.check_output(
                ["sysctl", "-n", "machdep.cpu.brand_string"], text=True).strip()
        with open("/proc/cpuinfo") as archivo:
            for linea in archivo:
                if linea.startswith("model name"):
                    return linea.split(":", 1)[1].strip()
    except Exception:
        pass
    return platform.processor() or "No disponible"


# Hardware
memoria = psutil.virtual_memory()
print(f"CPU: {nombre_cpu()}")
print(f"Núcleos: {psutil.cpu_count(logical=False)} físicos, {psutil.cpu_count(logical=True)} lógicos")
print(f"RAM: {memoria.total / 1024**3:.1f} GB (disponible al iniciar: {memoria.available / 1024**3:.1f} GB)")

# Software
print(f"Sistema operativo: {platform.system()} {platform.release()} ({platform.machine()})")
print(f"Python: {platform.python_version()}")
for libreria in ["faker", "psutil", "numpy", "pandas", "matplotlib"]:   # ajusta a las que uses
    try:
        print(f"  {libreria}: {metadata.version(libreria)}")
    except metadata.PackageNotFoundError:
        print(f"  {libreria}: no instalada")

# Condiciones de la medición 
print("Condiciones: portátil, [conectado a corriente: sí], modo de energía [Mayor rendimiento], "
      "programas abiertos: [VS Code]")

# Versiones exactas de todo lo instalado, para reconstruir el entorno
with open("requirements.txt", "w", encoding="utf-8") as archivo:
    archivo.write(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))

CPU: AMD Ryzen 5 5500U with Radeon Graphics
Núcleos: 6 físicos, 12 lógicos
RAM: 7.3 GB (disponible al iniciar: 1.8 GB)
Sistema operativo: Windows 11 (AMD64)
Python: 3.14.7
  faker: 40.1.2
  psutil: 7.2.2
  numpy: 2.5.2
  pandas: 3.0.6
  matplotlib: 3.11.2
Condiciones: portátil, [conectado a corriente: sí], modo de energía [Mayor rendimiento], programas abiertos: [VS Code]


## 4. Parámetros globales

Los valores de esta sección fueron **sugeridos con asistencia de IA** (Claude, Anthropic).
[Si los ajustas después de probar, escribe qué cambiaste y por qué.]
Se definen en una sola celda para que cualquiera pueda modificarlos y repetir los experimentos.
Cómo se generan los datos, cómo se mide y qué estadísticas se calculan se explica en la
sección 5 (Metodología).

| Parámetro | Valor | Por qué |
|---|---|---|
| Tamaños de datos (N) | 100, 1.000, 10.000, 100.000, 1.000.000 | Cada tamaño es 10 veces el anterior. Esto cubre cuatro órdenes de magnitud con pocos puntos. Incluye 10.000, el caso del enunciado. |
| Búsquedas por medición (M) | 1.000 | Una sola búsqueda es demasiado rápida para medirla con fiabilidad, mil hacen que el tiempo total sea medible incluso con N pequeño. |
| Búsquedas exitosas | 80 % | La mayoría busca IDs que existen, y el 20 % restante busca IDs inexistentes, para incluir también el costo de una búsqueda fallida. |
| Repeticiones | 30 | Suficientes para calcular promedio y desviación estándar. |
| Calentamiento | 3 ejecuciones descartadas | Las primeras ejecuciones suelen ser más lentas y no son representativas. |
| Semilla base | 1000 | La repetición *i* usa la semilla de aleatorización `1000 + i`, de modo que cada medición utiliza datos distintos. |
| Orden del árbol B+ | 4 | Es el orden que se usó al implementarlo. |
| Tamaño del rango (k) | 50 | Cada búsqueda por rango devuelve exactamente 50 estudiantes, sin importar el tipo de datos. |



In [22]:
# Parámetros globales (valores sugeridos con asistencia de IA: Claude, Anthropic, octubre 2026)
TAMANOS_N = [100, 1_000, 10_000, 100_000, 1_000_000]
M_BUSQUEDAS = 1_000
PROPORCION_EXITOSAS = 0.8
REPETICIONES = 30
CALENTAMIENTO = 3
SEMILLA_BASE = 1000
ORDEN_BPLUS = 4
K_RANGO = 50

## 5. Metodología

> Sección redactada con asistencia de IA (Claude, Anthropic) y revisada por mí.

Esta sección explica cómo se mide el **tiempo de ejecución** de cada operación al probar tres estructuras de datos (Lista, Árbol Binario de Búsqueda o ABB, y Árbol B+), variando la cantidad de datos ($N$) y su orden (aleatorio u ordenado).

### 5.1 Entorno de pruebas (Hardware y software)

Las mediciones se realizan en un único equipo de cómputo. Los tiempos en milisegundos o segundos pueden cambiar si el código se ejecuta en otra computadora más rápida o lenta, pero la tendencia de crecimiento y las comparaciones entre algoritmos se mantendrán iguales.


### 5.2 Tamaños de entrada

Probamos con cinco tamaños de datos: **$N = 100$, $1.000$, $10.000$, $100.000$ y $1.000.000$** estudiantes.

* **Por qué se eligen estos valores:** Cada número es 10 veces mayor que el anterior. Esto permite evaluar desde volúmenes muy pequeños hasta un millón de registros. Al graficar en una escala logarítmica (donde los ejes avanzan multiplicándose por 10), estos puntos quedan separados a la misma distancia, lo que facilita ver cómo aumenta el tiempo al crecer $N$.
* **Medición por lotes ($M = 1.000$):** Medir el tiempo de buscar un solo estudiante es impredecible porque toma fracciones de microsegundo. Para solucionarlo, en cada prueba se ejecutan **$M = 1.000$ búsquedas seguidas** y se mide el tiempo total del lote.

### 5.3 Generación de los datos

Generamos los datos de los estudiantes en dos escenarios para evaluar el comportamiento bajo distintas condiciones:

| Tipo de datos | Cómo se generan los IDs | Por qué se utiliza |
| --- | --- | --- |
| **Aleatorios** | $N$ identificadores únicos elegidos al azar entre $1$ y $10 \cdot N$. | Representa el uso normal. En el ABB, los datos desordenados producen un árbol balanceado (con alturas similares en sus ramas). |
| **Ordenados** | Identificadores en secuencia estricta: $1, 2, 3, \dots, N$. | Representa el peor escenario para el ABB. Insertar datos ordenados hace que el árbol no se ramifique y se transforme en una lista larga. |

El rango hasta $10 \cdot N$ deja "huecos" entre los identificadores, lo que permite probar qué pasa cuando se busca un registro que no existe. Cada repetición utiliza una **semilla numérica** (un número inicial para el generador de azar), lo que garantiza que los datos generados sean exactamente los mismos si se vuelve a correr el código.


### 5.4 Generación de las búsquedas

Para comparar las estructuras de forma justa, en cada repetición la Lista, el ABB y el Árbol B+ reciben **exactamente la misma información y las mismas consultas**:

* **Búsqueda por ID puntual:** Se busca un lote de $M = 1.000$ IDs. El 80% de las consultas son de estudiantes que sí existen en la estructura y el 20% son de IDs inexistentes, evaluando ambos casos.
* **Búsqueda por rango:** Se buscan $M = 1.000$ rangos. Para que la comparación sea justa, cada rango se define para recuperar **exactamente $k = 50$ estudiantes**. De este modo, todas las estructuras realizan el mismo trabajo de lectura.

### 5.5 Control y medición del tiempo

Para obtener tiempos precisos y libres de interferencias externas, aplicamos los siguientes mecanismos:

* **Reloj de alta precisión (`time.perf_counter_ns()`):** Lee el tiempo directamente del procesador en nanosegundos ($1\text{ segundo} = 1.000.000.000\text{ nanosegundos}$).
* **Aislamiento de la prueba:** El reloj solo mide la ejecución del algoritmo de búsqueda o inserción. La creación previa de los datos en memoria queda fuera del cronómetro.
* **Desactivación del recolector de basura (`gc.disable()`):** Python limpia la memoria no utilizada de forma automática. Desactivamos este proceso durante la medición para evitar que una limpieza espontánea infle el tiempo de una prueba.
* **Fase de calentamiento:** Se ejecutan 3 repeticiones previas que se descartan. Esto permite que el procesador cargue el código en su memoria rápida (caché) antes de tomar las mediciones definitivas.

### 5.6 Repeticiones y reproducibilidad

Cada configuración (por ejemplo, ejecutar un lote de $M = 1.000$ búsquedas en un ABB poblado con $N = 10.000$ estudiantes con datos aleatorios) se repite 30 veces.

* **Por qué 30 veces:** El tiempo de ejecución varía levemente en cada intento debido a tareas en segundo plano del sistema operativo. Al tomar 30 mediciones, el promedio resulta estadísticamente confiable (por el Teorema del Límite Central, que establece que el promedio de 30 o más muestras se comporta como una distribución normal suave).
* **Guardado automático:** Cada repetición guarda su resultado directamente en un archivo CSV. Esto permite reanalizar los datos o cambiar las gráficas en cualquier momento sin tener que repetir los experimentos.

### 5.7 Fórmulas estadísticas utilizadas

Para sintetizar las 30 repeticiones de cada prueba se aplican las siguientes fórmulas:

#### 1. Media ($\bar{x}$)

Es el promedio aritmético de los tiempos de ejecución.

$$\bar{x} = \frac{\sum_{i=1}^{n} t_i}{n}$$

* **Aplicación:** Es el valor representativo que se grafica en el eje vertical de tiempo.

#### 2. Desviación Estándar Muestral ($s$)

Mide qué tan dispersos están los tiempos individuales respecto al promedio.

$$s = \sqrt{\frac{\sum_{i=1}^{n} (t_i - \bar{x})^2}{n - 1}}$$

* **Aplicación:** Indica la estabilidad del sistema. Una desviación alta significa que los tiempos variaron mucho entre una repetición y otra debido a interferencias externas.

#### 3. Mediana y Rango Intercuartílico ($IQR$)

* **Mediana:** Es el dato que queda en el centro al ordenar las 30 mediciones de menor a mayor.
* **IQR ($Q_3 - Q_1$):** Es la diferencia entre el cuartil 3 (el valor que supera al 75% de los datos) y el cuartil 1 (el valor que supera al 25%). Representa el ancho donde se concentra el 50% central de las mediciones.

#### 4. Intervalo de Confianza al 95% (IC 95%)

Calcula el rango dentro del cual se estima la media verdadera del algoritmo con un 95% de confianza.

$$\text{IC 95\%} = \bar{x} \pm \left( t_{0.025, \, n-1} \cdot \frac{s}{\sqrt{n}} \right)$$

Donde:

* $\bar{x}$ es la media muestral.
* $s$ es la desviación estándar.
* $n$ es el número de repeticiones ($n = 30$).
* $t_{0.025, \, n-1}$ es el factor de la distribución $t$ de Student para $29$ grados de libertad ($\approx 2.045$).

**Cómo se aplica en las gráficas:** Se dibuja como barras de error verticales sobre cada punto. Si las barras de dos estructuras no se cruzan (no se solapan), se confirma visualmente que una estructura es más rápida que la otra con validez estadística.

### 5.8 Identificación de valores atípicos (Criterio de Tukey)

Un tiempo atípico (o *outlier*) es una medición anormalmente alta producida por una pausa del sistema operativo o un proceso en segundo plano, y no por una lentitud real del algoritmo.

Para detectar estos picos, definimos dos límites numéricos:

$$\text{Límite Inferior} = Q_1 - 1.5 \cdot IQR$$

$$\text{Límite Superior} = Q_3 + 1.5 \cdot IQR$$

Cualquier tiempo medido que caiga por debajo del límite inferior o por encima del límite superior se considera **atípico**.

* **Aplicación en el proyecto:** Las funciones procesan los datos de dos formas simultáneas: con todos los datos y descartando los atípicos (guardados con el sufijo `_sin`). Reportar ambas opciones garantiza total transparencia en las conclusiones.

### 5.9 Comparación con la complejidad teórica (Pendiente Log-Log)

La teoría indica que el tiempo $T(N)$ de un algoritmo según el tamaño $N$ sigue una ley de potencia:

$$T(N) \approx a \cdot N^b$$

Donde $b$ representa el exponente de complejidad ($b = 1$ para lineal, $b = 2$ para cuadrático, etc.). Al aplicar logaritmo en base 10 a ambos lados de la ecuación, obtenemos una línea recta:

$$\log_{10}(T(N)) = \log_{10}(a) + b \cdot \log_{10}(N)$$

Esta ecuación tiene la forma lineal $y = m \cdot x + c$, donde la **pendiente $m$ es igual al exponente $b$**.

Calculamos la pendiente mediante una regresión lineal sobre los logaritmos de $N$ y del tiempo promedio. La interpretación del resultado es la siguiente:

| Pendiente ($b$) | Tipo de crecimiento | Comportamiento esperado en las estructuras |
| --- | --- | --- |
| **$\approx 0$** | **Constante $O(1)$ / Logarítmico $O(\log N)$** | El tiempo casi no sube al aumentar $N$. Esperado en la búsqueda puntual dentro del Árbol B+ y el ABB balanceado. |
| **$\approx 1$** | **Lineal $O(N)$** | Si $N$ se multiplica por 10, el tiempo sube 10 veces. Esperado en la búsqueda sobre la Lista y en las búsquedas por rango. |
| **$\approx 2$** | **Cuadrático $O(N^2)$** | Si $N$ se multiplica por 10, el tiempo sube 100 veces ($10^2$). Esperado al construir un ABB insertando datos ya ordenados. |

## 6. Funciones de utilidad

In [23]:
# Funciones de apoyo para resumir los tiempos, guardarlos y graficarlos.
# Escritas con asistencia de IA (Claude, Anthropic, octubre 2026), según lo
# permitido por el enunciado del Laboratorio 3.

import os
import numpy as np                 # cálculos con listas de números
import pandas as pd                # tablas (el CSV se lee y se guarda con pandas)
import matplotlib.pyplot as plt    # gráficas
from scipy import stats            # solo se usa para un número: el factor t del intervalo de confianza

def calcular_estadisticas(tiempos):
    """
    Calcula las estadísticas de UNA lista de tiempos (por ejemplo, las 30
    repeticiones de "ABB con N=10.000").
    Devuelve un diccionario con: n, media, desviación, mediana, iqr, ic_inf, ic_sup.
    (IQR = ancho del 50 % central de los datos. IC = intervalo de confianza del 95 %:
    el rango donde probablemente está la media verdadera.)
    """
    t = np.array(tiempos, dtype=float)
    n = len(t)

    media = t.mean()
    desviacion = t.std(ddof=1)             # ddof=1: desviación "muestral" (divide entre n-1)
    mediana = np.median(t)
    q1, q3 = np.percentile(t, [25, 75])    # cuartiles: valores que dejan 25 % y 75 % de los datos abajo

    # Margen del intervalo de confianza: factor t * desviación / raíz de n
    factor_t = stats.t.interval(0.95, df=n - 1)[1]
    margen = factor_t * desviacion / np.sqrt(n)

    return {
    "n": n, 
    "media": media, 
    "desviacion": desviacion, 
    "mediana": mediana,
    "iqr": q3 - q1, 
    "ic_inf": media - margen, 
    "ic_sup": media + margen
    }


def resumir(tiempos):
    """
    Resume los tiempos de UNA configuración, de dos maneras:
      - con todos los datos (nombres normales: media, desviacion, ...)
      - sin los atípicos (mismos nombres con sufijo "_sin": media_sin, ...)
    Un dato es atípico si queda fuera de [Q1 - 1,5*IQR, Q3 + 1,5*IQR] (criterio de Tukey).
    También devuelve cuántos atípicos hubo ("atipicos").
    """
    t = np.array(tiempos, dtype=float)

    # Límites para detectar atípicos
    q1, q3 = np.percentile(t, [25, 75])
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    # True en los datos normales, False en los atípicos (máscara booleana, un arreglo paralelo de booleanos
    # por cada valor de tiempo)
    es_normal = (t >= limite_inferior) & (t <= limite_superior)

    resumen = calcular_estadisticas(t)                      # con todos los datos (incluyendo atípicos)
    resumen["atipicos"] = int((~es_normal).sum())           # cuántos atípicos hay

    for nombre, valor in calcular_estadisticas(t[es_normal]).items():   # sin los atípicos
        resumen[nombre + "_sin"] = valor
    return resumen

def resumir_tabla(datos, columna_curva, columna_x):
    """
    Aplica resumir() a cada configuración de la tabla de mediciones crudas.
    "datos" debe tener una columna "tiempo_ns" (un tiempo por fila).
      - columna_curva: columna que define cada línea de la gráfica (ej. "estructura")
      - columna_x: columna del eje horizontal (ej. "N")
    Devuelve una tabla con una fila por cada pareja (curva, x).
    """
    filas = []
    for (curva, x), grupo in datos.groupby([columna_curva, columna_x]):
        fila = {columna_curva: curva, columna_x: x}
        fila.update(resumir(grupo["tiempo_ns"]))
        filas.append(fila)
    return pd.DataFrame(filas)

def guardar_csv(datos, nombre):
    """
    Guarda la tabla de mediciones en resultados/<nombre>.csv (crea la carpeta si no existe).
    Para volver a leerla más tarde: pd.read_csv("resultados/<nombre>.csv")
    """
    os.makedirs("resultados", exist_ok=True)
    ruta = f"resultados/{nombre}.csv"
    datos.to_csv(ruta, index=False)
    return ruta

def graficar(resumen, columna_curva, columna_x, titulo, etiqueta_x,
             unidad="ms", sin_atipicos=False, escala_log=True):
    """
    Dibuja una línea por cada valor de `columna_curva`: la media del tiempo en cada x,
    con barras verticales que muestran el intervalo de confianza del 95 %.
    `resumen` es lo que devuelve resumir_tabla().
      - unidad: "ns", "us", "ms" o "s" (unidad del eje vertical)
      - sin_atipicos: False = usa todos los datos; True = usa los datos sin atípicos
      - escala_log: True = ambos ejes en escala logarítmica (para leer la pendiente)
    """
    divisor = {"ns": 1, "us": 1e3, "ms": 1e6, "s": 1e9}[unidad]   # los tiempos están en ns
    sufijo = "_sin" if sin_atipicos else ""

    plt.figure(figsize=(8, 5))
    for curva, tabla in resumen.groupby(columna_curva):
        tabla = tabla.sort_values(columna_x)
        media = tabla["media" + sufijo] / divisor
        abajo = media - tabla["ic_inf" + sufijo] / divisor      # largo de la barra hacia abajo
        arriba = tabla["ic_sup" + sufijo] / divisor - media     # largo de la barra hacia arriba
        plt.errorbar(tabla[columna_x], media, yerr=[abajo, arriba],
                     marker="o", capsize=3, label=curva)

    if escala_log:
        plt.xscale("log")
        plt.yscale("log")
    plt.title(titulo + (" (sin atípicos)" if sin_atipicos else ""))
    plt.xlabel(etiqueta_x)
    plt.ylabel(f"Tiempo medio ({unidad})  |  barras: IC 95 %")
    plt.legend(title=columna_curva)
    plt.grid(True, which="both", alpha=0.3)
    plt.show()

def pendiente_loglog(x, tiempo):
    """
    Calcula la pendiente de la recta que mejor se ajusta a log10(tiempo) contra log10(x).
    Esa pendiente es el exponente de crecimiento:
    cerca de 0 = casi constante o logarítmico, cerca de 1 = lineal, cerca de 2 = cuadrático.
    """
    # 1. Convierte las entradas N y los tiempos a escala logarítmica
    x_log = np.log10(x) # (Los distintos tamaños de entrada)
    y_log = np.log10(tiempo) # (Los tiempos de cada tamaño de entrada)

    # 2. Ajusta una recta de grado 1 (y = m*x + b) usando mínimos cuadrados ordinarios
    #    Devuelve una lista con dos elementos: [pendiente, interseccion]
    resultado = np.polyfit(x_log, y_log, 1)

    # 3. Toma únicamente la pendiente m (descartando b mediante _)
    pendiente, _ = resultado

    return pendiente

# 2. Experimentos

## Experimento 0

> Nota: Sección redactada con asistencia de IA (Gemini) y revisada por Daniel Sánchez Escobar.

**Pregunta:** ¿Las tres estructuras de datos (Lista, ABB y Árbol B+) funcionan correctamente y devuelven los mismos resultados en las operaciones de buscar, insertar, listar y buscar por rango?
**Hipótesis:** Sí. Todas las estructuras deben comportarse de forma idéntica respecto a los datos devueltos. Cualquier diferencia indica un error de programación que debe corregirse antes de medir tiempos en los siguientes experimentos.

### Diseño:

Para cada prueba se construyen la Lista, el ABB y el Árbol B+ insertando exactamente los mismos datos de estudiantes. Luego se verifica que:

1. Los métodos "listar" de cada implementación coincidan exactamente en cuánto a los ID's de los estudiantes, tanto en existencia como en orden.
2. Los métodos "buscar" de cada implementación encuentren a un mismo estudiante puntual por su ID.
3. Que el método "buscar" indique correctamente que un estudiante no fue encontrado para ID's inexistentes.
4. Los métodos "buscar_rango_id" devuelvan exactamente los registros dentro del rango solicitado, respondiendo bien en casos límite (rangos de un solo elemento, rangos vacíos o rangos invertidos).
5. Para los árboles, la función "altura" devuelva 0 únicamente cuando la estructura no tiene elementos ($N = 0$), y que en el ABB coincida exactamente con $N$ cuando los datos entran ordenados o de forma descendente.

#### Variables del experimento
| Variable | Valores probados |
|---|---|
| Cantidad de datos (N) | 0, 1, 2, 3, 10, 100, 1.000 |
| Orden de inserción | aleatorio, ascendente, descendente |
| Orden del Árbol B+ | 4, 5, 8 |
| Semillas aleatorias | 1000 a 1004 |

In [24]:
import re
import pandas as pd
from ABB import ABB
from ArbolBPlus import ArbolBPlus
from GestorListaEstudiantes import GestorListaEstudiantes
from funciones_auxiliares import generar_estudiantes

def extraer_ids(texto: str) -> list:
    """
    Busca en el texto devuelto por las estructuras y extrae únicamente los números de ID.
    Sirve para comparar rápidamente si las funciones listar() o buscar_rango_id() 
    están devolviendo los IDs correctos y en el orden esperado.
    """
    return [int(x) for x in re.findall(r"id : (\d+)", texto)]

def texto_busqueda_esperado(estudiante: dict) -> str:
    """Construye exactamente el mismo texto que debería devolver una búsqueda exitosa."""
    return "Encontrado :\n" + "".join(f"{k} : {v} | " for k, v in estudiante.items())

def validar_estructuras(n: int, tipo: str, orden_bplus: int, semilla: int) -> list:
    """
    Construye la Lista, el ABB y el B+ con 'n' estudiantes y prueba cada operación.
    Devuelve una lista con los fallos encontrados. Si todo está bien, devuelve una lista vacía.
    """
    # 1. Preparar los datos de prueba dependiendo del orden elegido
    estudiantes_crudos = generar_estudiantes(n, ordenados=(tipo != "aleatorio"), semilla=semilla)
    if tipo == "descendente":
        estudiantes_crudos = estudiantes_crudos[::-1] # Esta acción invierte la lista.
    
    ids_esperados = sorted([e["id"] for e in estudiantes_crudos])
    
    # 2. Inicializar las tres estructuras
    estructuras = {
        "Lista": GestorListaEstudiantes([]),
        "ABB": ABB(),
        f"B+ (orden {orden_bplus})": ArbolBPlus(orden_bplus),
    }

    fallos = []

    # 3. Evaluar cada estructura paso a paso
    for nombre, estructura in estructuras.items():
        
        # Prueba A: Insertar los estudiantes
        for e in estudiantes_crudos:
            estructura.insertar(e)

        # Prueba B: Listar (debe devolver todos los IDs en orden ascendente)
        if extraer_ids(estructura.listar()) != ids_esperados:
            fallos.append((nombre, "listar no ordenó bien o faltan datos"))

        # Prueba C: Buscar estudiantes que SÍ existen
        for e in estudiantes_crudos:
            if estructura.buscar(e["id"]) != texto_busqueda_esperado(e):
                fallos.append((nombre, "buscar falló con estudiante existente"))

        # Prueba D: Buscar estudiantes que NO existen
        ids_inexistentes = [0, 999999, 999998] # IDs que sabemos que no se generaron
        for id_falso in ids_inexistentes:
            esperado = f"Estudiante con id: {id_falso} no fue encontrado."
            if estructura.buscar(id_falso) != esperado:
                fallos.append((nombre, "buscar falló con estudiante inexistente"))

        # Prueba E: Validar la altura (si la estructura tiene ese método)
        if hasattr(estructura, "altura"):
            h = estructura.altura()
            if n == 0 and h != 0:
                fallos.append((nombre, "altura no es 0 en estructura vacía"))
            # En un ABB insertado en orden, la altura debe ser igual a N
            if nombre == "ABB" and tipo != "aleatorio" and h != n:
                fallos.append((nombre, "altura incorrecta en ABB degenerado"))

        # Prueba F: Insertar un dato nuevo después de haber llenado la estructura
        estudiante_nuevo = {"id": 888888, "nombre": "Prueba", "edad": 20, "promedio": 4.0}
        estructura.insertar(estudiante_nuevo)
        
        if estructura.buscar(estudiante_nuevo["id"]) != texto_busqueda_esperado(estudiante_nuevo):
            fallos.append((nombre, "no se pudo buscar el estudiante recién insertado"))

        # Prueba G: búsquedas por rango
        if ids_esperados:
            casos_rango = [
                (ids_esperados[0], ids_esperados[-1]),             # 1. Rango completo
                (ids_esperados[0], ids_esperados[0]),              # 2. Un solo ID
                (0, ids_esperados[-1] + 100),                      # 3. Rango amplio que cubre todo
                (ids_esperados[-1] + 10, ids_esperados[-1] + 50), # 4. Rango fuera de los datos (vacío)
                (50, 10)                                           # 5. Límite inferior mayor que el superior (lb > ub)
            ]
        else:
            casos_rango = [(1, 10)]  # Caso especial cuando N = 0 (estructura vacía)

        for lb, ub in casos_rango:
            # Generamos la lista matemática con los IDs que DEBERÍAN estar dentro del rango [lb, ub]
            ids_rango_esperados = [i for i in ids_esperados if lb <= i <= ub]
            
            # Consultamos la estructura
            texto_rango = estructura.buscar_rango_id(lb, ub)
            
            # Comparamos únicamente los IDs devueltos contra los esperados
            if extraer_ids(texto_rango) != ids_rango_esperados:
                fallos.append((nombre, f"buscar_rango_id [{lb}, {ub}]"))

    return fallos

In [25]:
N_E0 = [0, 1, 2, 3, 10, 100, 1000] # Tamaños de entrada
fallos_totales = []
combinaciones = 0

for n in N_E0: # Para cada tamaño de entrada
    for tipo in ["aleatorio", "ordenado", "descendente"]: # Para cada configuración de los datos
        for orden in [4, 5, 8]: # Para cada orden (para el árbol B+)
            for semilla in range(1000, 1005): # Semillas fijas de aleatorización
                combinaciones += 1
                errores = validar_estructuras(n, tipo, orden, semilla)
                
                # Registrar detalles de cada fallo encontrado
                for estructura, prueba in errores:
                    fallos_totales.append({
                        "N": n, 
                        "tipo": tipo, 
                        "orden_B+": orden, 
                        "estructura": estructura, 
                        "prueba": prueba
                    })

print(f"Total de combinaciones probadas: {combinaciones}")
print(f"Total de fallos encontrados: {len(fallos_totales)}")

if fallos_totales:
    df_fallos = pd.DataFrame(fallos_totales)
    display(df_fallos.groupby(["estructura", "prueba"]).size())
else:
    print("¡Éxito! Todas las estructuras pasaron las pruebas de correctitud.")

Total de combinaciones probadas: 315
Total de fallos encontrados: 0
¡Éxito! Todas las estructuras pasaron las pruebas de correctitud.


### Análisis y conclusión del experimento 0
Se evaluaron exitosamente las 315 combinaciones planificadas, abarcando variaciones en la cantidad de datos ($N \in \{0, 1, 2, 3, 10, 100, 1.000\}$), el orden de entrada (aleatorio, ascendente y descendente), el orden del Árbol B+ ($n \in \{4, 5, 8\}$) y distintas semillas de aleatorización. Ninguna de las tres estructuras presentó fallos en las pruebas funcionales de inserción, búsqueda individual, listado completo, cálculo de altura (solo en árboles) ni búsquedas por rango. Dado que no se registraron errores de lógica ni inconsistencias en los datos retornados, se concluye que la Lista, el ABB y el Árbol B+ implementan de forma correcta las operaciones requeridas y responden de manera funcionalmente equivalente. Las tres estructuras se encuentran en un estado estable y listas para ser sometidas a las pruebas de rendimiento temporal y espacio en los experimentos posteriores.

#### Limitaciones
1. Cobertura no exhaustiva de casos: el hecho de que todas las pruebas hayan sido superadas no constituye una demostración formal de la ausencia total de errores (bugs), sino únicamente que no se detectaron fallos dentro de los escenarios evaluados.
2. Tamaño máximo de entrada: la validación funcional se limitó a un máximo de $N = 1.000$ elementos. Aunque es suficiente para verificar la lógica de punteros, reestructuraciones y divisiones de nodos, no evalúa comportamientos bajo cargas masivas.